In [1]:
import torch
import torchvision
import torchvision.transforms as transforms


if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps")

print(device)

cuda:0


In [2]:
Unet = torch.hub.load('milesial/Pytorch-UNet', 'unet_carvana', pretrained=True, scale=0.5)

c:\Python38\lib\site-packages\torch\hub.py:294: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(
Downloading: "https://github.com/milesial/Pytorch-UNet/zipball/master" to C:\Users\Szymon/.cache\torch\hub\master.zip
Downloading: "https://github.com/milesial/Pytorch-UNet/releases/download/v3.0/unet_carvana_scale0.5_epoch2.pth" to C:\Users\Szymon/.cache\torch\hub\checkpoints\unet_carvana_scale0.5_epoch2.pth
100%|██████████| 118M/118M [00:34<00:00, 3.62MB/s] 


In [3]:
Unet

UNet(
  (inc): DoubleConv(
    (double_conv): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (down1): Down(
    (maxpool_conv): Sequential(
      (0): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (1): DoubleConv(
        (double_conv): Sequential(
          (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
 

In [ ]:
# model = torch.hub.load('mateuszbuda/brain-segmentation-pytorch', 'unet', in_channels=3, out_channels=1, init_features=32, pretrained=True)

In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class MyDataset(Dataset):
    def __init__(self, image_paths, target_paths, transform=None):
        self.image_paths = image_paths
        self.target_paths = target_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx])
        target = Image.open(self.target_paths[idx])

        if self.transform:
            image = self.transform(image)
            target = self.transform(target)

        return image, target

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.transforms import Compose

def train_unet(model, train_loader, criterion, optimizer, num_epochs=10):
    # Ustaw tryb modelu na trenowanie
    model.train()
    
    # Pętla trenowania przez zadaną liczbę epok
    for epoch in range(num_epochs):
        running_loss = 0.0
        
        # Pętla po wsadach danych
        for inputs, targets in train_loader:
            # Wyzeruj gradienty
            optimizer.zero_grad()
            
            # Przekształć dane wejściowe i docelowe na tensor PyTorch
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            # Przejdź przez model
            outputs = model(inputs)
            
            # Oblicz stratę
            loss = criterion(outputs, targets)
            
            # Propagacja wsteczna i aktualizacja wag
            loss.backward()
            optimizer.step()
            
            # Aktualizuj bieżący biegowy loss
            running_loss += loss.item() * inputs.size(0)
        
        # Oblicz średni loss dla danej epoki
        epoch_loss = running_loss / len(train_loader.dataset)
        
        # Wydrukuj postęp trenowania
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')
    
    print('Training finished')

# Przygotuj dane do treningu
transform = transforms.Compose([
    transforms.ToTensor()  # Konwertuj obrazy na tensory
])
image_paths_train = []
for image_name in os.listdir('images_train/input'):
    image_paths_train.append(f'images_train/input/{image_name}')
target_paths_train = []
for target_name in os.listdir('images_train/output'):
    target_paths_train.append(f'images_train/output/{target_name}')
image_paths_test = []
for image_name in os.listdir('images_test/input'):
    image_paths_test.append(f'images_test/input/{image_name}')
target_paths_test = []
for target_name in os.listdir('images_test/output'):
    target_paths_test.append(f'images_test/output/{target_name}')
batch_size = 16

train_dataset = MyDataset(image_paths_train, target_paths_train, transform=transform)
test_dataset = MyDataset(image_paths_test, target_paths_test, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


# Określ funkcję straty i optymalizator
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(Unet.parameters(), lr=0.001)

# Rozpocznij trenowanie
train_unet(Unet, train_loader, criterion, optimizer, num_epochs=10)


In [ ]:
import time
import copy


def train_model(
    model, dataloaders, criterion, optimizer, num_epochs=25
):
    since = time.time()

    val_acc_history = []

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(1, num_epochs + 1):
        print("Epoch {}/{}".format(epoch, num_epochs))
        print("-" * 10)

        # Each epoch has a training and validation phase
        for phase in ["train", "val"]:
            if phase == "train":
                model.train()  # Set model to training mode
            else:
                model.eval()  # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data.
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # zero the parameter gradients
                optimizer.zero_grad()

                # forward
                # track history if only in train
                with torch.set_grad_enabled(phase == "train"):
                    # Get model outputs and calculate loss

                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)

                    # backward + optimize only if in training phase
                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                # statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.float() / len(dataloaders[phase].dataset)

            print("{} Loss: {:.4f} Acc: {:.4f}".format(phase, epoch_loss, epoch_acc))

            # deep copy the model
            if phase == "val" and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
            if phase == "val":
                val_acc_history.append(epoch_acc)

        print()

    time_elapsed = time.time() - since
    print(
        "Training complete in {:.0f}m {:.0f}s".format(
            time_elapsed // 60, time_elapsed % 60
        )
    )
    print("Best val Acc: {:4f}".format(best_acc))

    # load best model weights
    model.load_state_dict(best_model_wts)
    return model, val_acc_history

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import ImageFolder
from torchvision.transforms import transforms
from torch.utils.data import DataLoader


data_dir = "students_images"
batch_size = 32

model = Unet.to(device)

transform = transforms.Compose([
    transforms.ToTensor()  # Konwertuj obrazy na tensory
])


train_data_path = "images_train/input"
train_label_path = "images_train/output"
test_data_path = "images_test/input"
test_label_path = "images_test/output"

train_iter = DataLoader(
    ImageFolder(train_data_path, transform=transform),
    batch_size=batch_size,
    shuffle=True
)

test_iter = DataLoader(
    ImageFolder(test_data_path, transform=transform),
    batch_size=batch_size
    shuffle=True
)



train_dataset = DataLoader(train_data_path, train_label_path, batch_size=batch_size, transform=transform, shuffle=True)



In [ ]:
def train_fine_tuning(net, learning_rate, num_epochs=15):

    trainer = torch.optim.SGD([{"params": Unet.parameters(), "lr": learning_rate*10}], lr=learning_rate) # your code here

    dataloaders_dict = {"train": train_iter, "val": test_iter}
    criterion = nn.CrossEntropyLoss()
    model_ft, hist = train_model(
        net, dataloaders_dict, criterion, trainer, num_epochs=num_epochs
    )
    return model_ft, hist
